In [1]:
import pdfplumber
import os
import pandas as pd
import re

In [2]:
def year_month_key(filename):
    year, month = map(int, re.search(r"Decision_(\d+)_(\d+)_", filename).groups())
    return year, month

def merge_pdfs_to_single_txt(source_folder, output_txt_name):

    file_names = [x for x in os.listdir("Reports") if x.startswith("Decision_")]
    file_names = sorted(file_names, key=year_month_key)

    global df 

    df = pd.DataFrame(columns = ["ReportName", "Text"])
    index = 0

    for file_name in file_names:
        pdf_path = os.path.join(source_folder, file_name)
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                file_content = []
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        file_content.append(text)
                single_doc_text = "\n".join(file_content)
                single_doc_text = single_doc_text.replace("\n", " ")
                start_index = single_doc_text.index("The Monetary Policy Committee")
                df.loc[index] = [file_name, single_doc_text[start_index:]]
                index+=1
                print(f"Okundu: {file_name}")
                
        except Exception as e:
            print(f"Hata oluştu ({file_name}): {e}")

    df.to_excel("1_DecisionReportsV1.xlsx")
    print(f"\nBütün işlemler tamamlandı! Dosya konumu: {output_txt_name}")

merge_pdfs_to_single_txt("Reports", "Tum_Raporlar.txt")

Okundu: Decision_2006_1_January.pdf
Okundu: Decision_2006_2_February.pdf
Okundu: Decision_2006_3_March.pdf
Okundu: Decision_2006_4_April.pdf
Okundu: Decision_2006_5_May.pdf
Okundu: Decision_2006_6_June.pdf
Okundu: Decision_2006_7_June.pdf
Okundu: Decision_2006_8_July.pdf
Okundu: Decision_2006_9_August.pdf
Okundu: Decision_2006_10_September.pdf
Okundu: Decision_2006_11_October.pdf
Okundu: Decision_2006_12_November.pdf
Okundu: Decision_2006_13_December.pdf
Okundu: Decision_2007_1_January.pdf
Okundu: Decision_2007_2_February.pdf
Okundu: Decision_2007_3_March.pdf
Okundu: Decision_2007_4_April.pdf
Okundu: Decision_2007_5_May.pdf
Okundu: Decision_2007_6_June.pdf
Okundu: Decision_2007_7_July.pdf
Okundu: Decision_2007_8_August.pdf
Okundu: Decision_2007_9_September.pdf
Okundu: Decision_2007_10_October.pdf
Okundu: Decision_2007_11_November.pdf
Okundu: Decision_2007_12_December.pdf
Okundu: Decision_2008_1_January.pdf
Okundu: Decision_2008_2_February.pdf
Okundu: Decision_2008_3_March.pdf
Okundu: D

In [3]:
df = pd.read_excel("1_DecisionReportsV1.xlsx", index_col="Unnamed: 0")
df.head()

,ReportName,Text
0,Decision_2006_1_January.pdf,The Monetary Policy Committee has concluded th...
1,Decision_2006_2_February.pdf,The Monetary Policy Committee has decided to k...
2,Decision_2006_3_March.pdf,The Monetary Policy Committee has decided to k...
3,Decision_2006_4_April.pdf,The Monetary Policy Committee has decided to c...
4,Decision_2006_5_May.pdf,The Monetary Policy Committee has decided to k...


In [4]:
import nltk
import re

nltk.download('punkt')

SentenceDf = pd.DataFrame(columns = ["ReportName", "Sentences"])

def sentences_from_file(df):


    index = 0
    
    for row in df.index:

        reportname = df.loc[row, "ReportName"]
        full_text = df.loc[row, "Text"]
        sentences = nltk.sent_tokenize(full_text)
        
        for sentence in sentences:
            if len(sentence) < 20:
                continue
            SentenceDf.loc[index] = [reportname, sentence]
            index+=1

    print(f"Başarılı! Toplam {index} cümle kaydedildi.")
        
sentences_from_file(df)

SentenceDf.to_excel("1_DecisionReportsFinalDataset.xlsx")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Başarılı! Toplam 3474 cümle kaydedildi.


In [5]:
SentenceDf.head()

,ReportName,Sentences
0,Decision_2006_1_January.pdf,The Monetary Policy Committee has concluded th...
1,Decision_2006_1_January.pdf,It is estimated that although the downward tre...
2,Decision_2006_1_January.pdf,"Despite this downward trend, in the light of c..."
3,Decision_2006_1_January.pdf,The major reasons behind this argument are the...
4,Decision_2006_1_January.pdf,"As a result, it has been decided to keep the s..."


In [6]:
summary = (
    SentenceDf
    .groupby("Sentences")
    .agg(
        count=("ReportName", "size"),
        reports=("ReportName", lambda s: sorted(set(s)))
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

summary[summary["count"] > 9].head(3)
summary.to_excel("1_RepeatedSentences.xlsx")

In [7]:
# with open("1_SummaryReportsV3.txt", "w", encoding="utf-8") as f:
#     f.write("\n".join(SentenceDf2["Sentences"].astype(str)))